# Lesson 24 Lab — Gateway Admission, Rate Limits, and Multi-Tenancy

**Puzzle:** Should one cheap short request and one 32K batch job spend the same rate-limit unit?

This notebook retains the output of a complete RTX 5090 run.


## Why this matters

Request-count limits ignore prompt and output work. In a shared GPU service, one tenant can fill the queue or KV cache with a few large jobs while remaining under requests per minute.


## 0. Predict before running

1. Predict which policy admits more oversized batch work.
2. Choose a safe output reservation rule.
3. Define one fairness and one SLO gate.

For every answer, name the observation that would disprove it.


## 1. Name the concrete objects

A deterministic gateway simulation compares request-count and token-budget admission for interactive and batch tenants. It retains admitted work, rejects, per-tenant waiting, and fairness indices.

- Admission cost should approximate scarce resources.
- Rate, concurrency, and queue caps solve different abuse modes.
- Tenant identity must survive through metrics and audit without leaking secrets.


## 2. Derive the mechanism

A token-bucket can charge prompt tokens immediately and reserve an output allowance, then reconcile actual usage. Separate service classes and concurrency caps prevent large batch traffic from occupying every active slot. Authentication establishes tenant identity; authorization maps it to models, adapters, budgets, and logging policy.

### Mechanism at a glance

```mermaid
flowchart LR
  A["authenticated request"] --> P["prompt token charge"]
  P --> O["reserve output allowance"]
  O --> C{"quota + concurrency + route allowed?"}
  C -->|"yes"| Q["service-class queue"]
  C -->|"no"| R["bounded rejection"]
  Q --> V["vLLM pool"]
  V --> U["reconcile actual usage"]
```

### Walk it step by step

1. **Authenticate identity.** Bind a request to tenant, route, and policy.
2. **Estimate resource cost.** Charge prompt work and reserve an output budget.
3. **Apply layered limits.** Check rate, concurrent requests, queue depth, and service class.
4. **Reconcile usage.** Return unused allowance and audit actual token counts.


## 3. Inspect the execution environment

The next cell asserts CUDA, prints the RTX 5090/PyTorch/CUDA/vLLM identity, fixes a seed, and defines only the helpers used by this chapter.


In [1]:
LESSON_NO = 24
LESSON_TITLE = 'Gateway Admission, Rate Limits, and Multi-Tenancy'

from pathlib import Path
import gc, hashlib, importlib, inspect, ipaddress, json, math, os, random, re
import shutil, statistics, subprocess, sys, tempfile, time
from urllib.parse import urlparse

# The default FlashInfer sampler requires a local JIT link setup that is not
# guaranteed in wheel-only environments. vLLM's native PyTorch sampler keeps
# these labs reproducible without changing attention or scheduling backends.
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")

import requests
import torch
import vllm
import yaml
from vllm import LLM, SamplingParams

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260812 + LESSON_NO
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
MODEL_PATH = os.environ.get("CH3_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
MODEL = Path(MODEL_PATH)
assert MODEL.exists(), f"Set CH3_MODEL to a local model directory; not found: {MODEL}"

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name, "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__, "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0], "vllm": vllm.__version__,
    "model_path": MODEL.name, "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered: return float("nan")
    pos = (len(ordered) - 1) * q; lo, hi = math.floor(pos), math.ceil(pos)
    return ordered[lo] if lo == hi else ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def model_config():
    return json.loads((MODEL / "config.json").read_text(encoding="utf-8"))

def base_engine_args(**overrides):
    values = {
        "model": str(MODEL), "tokenizer": str(MODEL), "trust_remote_code": False,
        "dtype": "bfloat16", "max_model_len": 2048, "gpu_memory_utilization": 0.45,
        "enforce_eager": True, "seed": SEED, "max_num_seqs": 16,
    }
    values.update(overrides); return values

def output_record(item):
    completion = item.outputs[0]; tokens = list(completion.token_ids)
    return {
        "request_id": str(item.request_id), "prompt_tokens": len(item.prompt_token_ids or []),
        "output_tokens": len(tokens), "token_ids": tokens, "text_preview": completion.text[:120],
        "text_sha256": hashlib.sha256(completion.text.encode()).hexdigest(),
        "finish_reason": str(completion.finish_reason),
        "stop_reason": None if completion.stop_reason is None else str(completion.stop_reason),
        "num_cached_tokens": int(getattr(item, "num_cached_tokens", 0) or 0),
    }

def vllm_cli():
    candidate = Path(sys.executable).parent / "vllm"
    return str(candidate if candidate.exists() else (shutil.which("vllm") or "vllm"))

def cli_help(*args):
    result = subprocess.run([vllm_cli(), *args, "--help"], capture_output=True, text=True, timeout=60)
    return result.returncode, result.stdout + result.stderr

def run_server_probe(port, request_payload=None, scrape_metrics=False):
    log_path = Path(tempfile.gettempdir()) / f"ch03-vllm-{LESSON_NO}-{port}.log"
    command = [vllm_cli(), "serve", str(MODEL), "--host", "127.0.0.1", "--port", str(port),
               "--dtype", "bfloat16", "--max-model-len", "1024", "--gpu-memory-utilization", "0.45",
               "--enforce-eager", "--disable-uvicorn-access-log"]
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, text=True)
    ready = False
    try:
        deadline = time.time() + 300
        while time.time() < deadline:
            if process.poll() is not None: break
            try:
                if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                    ready = True; break
            except requests.RequestException: pass
            time.sleep(1)
        startup_s = time.perf_counter() - started
        if not ready:
            raise RuntimeError("vLLM server failed to start:\n" + log_path.read_text(errors="replace")[-6000:])
        models = requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=30)
        data = {"server_ready": True, "startup_s": startup_s,
                "models_status": models.status_code, "model_json": models.json()}
        if request_payload is not None:
            tick = time.perf_counter()
            chat = requests.post(f"http://127.0.0.1:{port}/v1/chat/completions",
                                 json=request_payload, timeout=180)
            data.update(chat_status=chat.status_code, chat_latency_s=time.perf_counter() - tick,
                        chat_json=chat.json())
        if scrape_metrics:
            response = requests.get(f"http://127.0.0.1:{port}/metrics", timeout=30)
            data.update(metrics_status=response.status_code, metrics_text=response.text)
        return data
    finally:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=30)
            except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=10)
        tail = log_path.read_text(errors="replace")[-4000:] if log_path.exists() else ""
        private_home = "/" + "root" + "/"
        globals()["SERVER_LOG_TAIL"] = tail.replace(str(MODEL), "$CH3_MODEL").replace(private_home, "<remote-home>/")


<remote-home>/vllm-ch03/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "vllm": "0.27.1",
  "model_path": "Qwen2.5-1.5B-Instruct",
  "seed": 20260836
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | equal request-count buckets |
| Candidate | prompt/output token budgets plus concurrency and service classes |
| Held constant | same arrivals, token estimates, engine capacity, and tenant weights |
| Measurements | admitted requests/tokens, rejection reasons, p95 wait, class isolation, and fairness |
| Evidence | `numerical-model` |

**Experiment:** Replay two tenant workloads through request-count and token-budget gateways.


## 5. Inspect the experiment code

The simulator keeps admission separate from GPU scheduling and records every decision. Its token costs are declared estimates rather than secret model calculations.

Do not execute until the code matches the frozen table.


In [2]:
events=[{"id":"i1","tenant":"interactive","arrival":0,"prompt":80,"output":40},
 {"id":"b1","tenant":"batch","arrival":0,"prompt":6000,"output":1000},
 {"id":"i2","tenant":"interactive","arrival":1,"prompt":120,"output":60},
 {"id":"b2","tenant":"batch","arrival":1,"prompt":8000,"output":1200},
 {"id":"i3","tenant":"interactive","arrival":2,"prompt":60,"output":30},
 {"id":"b3","tenant":"batch","arrival":3,"prompt":4000,"output":800},
 {"id":"i4","tenant":"interactive","arrival":4,"prompt":100,"output":50}]
def gateway(policy):
    count={"interactive":4,"batch":3}; budget={"interactive":1600,"batch":10500}
    admitted=[]; rejected=[]; used={"interactive":0,"batch":0}; waits={"interactive":[],"batch":[]}; clock={"interactive":0.,"batch":0.}
    for event in events:
        cost=event["prompt"]+event["output"]
        if policy=="request_count": allow=count[event["tenant"]]>0; count[event["tenant"]]-=int(allow)
        else: allow=budget[event["tenant"]]>=cost; budget[event["tenant"]]-=cost if allow else 0
        if not allow: rejected.append(event["id"]); continue
        start=max(event["arrival"],clock[event["tenant"]]); waits[event["tenant"]].append(start-event["arrival"])
        clock[event["tenant"]]=start+cost/(800 if event["tenant"]=="interactive" else 500)
        admitted.append(event); used[event["tenant"]]+=cost
    shares=list(used.values()); fairness=sum(shares)**2/(len(shares)*sum(x*x for x in shares)) if any(shares) else 0
    return {"admitted_requests":len(admitted),"admitted_tokens":sum(used.values()),
     "interactive_admitted":sum(x["tenant"]=="interactive" for x in admitted),
     "batch_admitted":sum(x["tenant"]=="batch" for x in admitted),"rejected":rejected,"tenant_tokens":used,
     "interactive_p95_wait":percentile(waits["interactive"],.95),
     "batch_p95_wait":percentile(waits["batch"],.95) if waits["batch"] else 0.,"fairness":fairness}
metrics={"request_count":gateway("request_count"),"token_budget":gateway("token_budget"),"events":events}
analysis=(f"Count admission accepted {metrics['request_count']['admitted_tokens']:,} tokens and "
          f"{metrics['request_count']['batch_admitted']} batch jobs; token budgeting accepted "
          f"{metrics['token_budget']['admitted_tokens']:,} and {metrics['token_budget']['batch_admitted']} "
          "while retaining interactive budget. Queue costs are modeled.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; vLLM 0.27.1.

| Measured field | Checked-in value |
|---|---:|
| Count-policy admitted tokens | 21,540 |
| Token-policy admitted tokens | 7,540 |
| Count-policy batch admits | 3 |
| Token-policy batch admits | 1 |
| Token-policy interactive p95 | 0.000000 |
| Token-policy fairness | 0.576686 |


## 7. Explain the result

Count admission accepted 21,540 tokens and 3 batch jobs; token budgeting accepted 7,540 and 1 while retaining interactive budget. Queue costs are modeled.

This interpretation is bounded to the printed model, GPU, packages, workload, and evidence label.


## 8. Keep the evidence label honest

This run is labeled **`numerical-model`**. A transparent allocator, scheduler, gateway, or policy model executed. It establishes the stated invariant, not native vLLM performance.

The next cell writes and prints the canonical JSON artifact.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 24, "title": 'Gateway Admission, Rate Limits, and Multi-Tenancy', "environment": ENV,
    "evidence_label": 'numerical-model', "metrics": metrics,
    "analysis": analysis, "conclusion": 'Token-aware admission better represents inference cost than request counts, but production limits require native traffic calibration.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 24,
  "title": "Gateway Admission, Rate Limits, and Multi-Tenancy",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "vllm": "0.27.1",
    "model_path": "Qwen2.5-1.5B-Instruct",
    "seed": 20260836
  },
  "evidence_label": "numerical-model",
  "metrics": {
    "request_count": {
      "admitted_requests": 7,
      "admitted_tokens": 21540,
      "interactive_admitted": 4,
      "batch_admitted": 3,
      "rejected": [],
      "tenant_tokens": {
        "interactive": 540,
        "batch": 21000
      },
      "interactive_p95_wait": 0.0,
      "batch_p95_wait": 27.759999999999998,
      "fairness": 0.5256972940341489
    },
    "token_budget": {
      "admitted_requests": 5,
      "admitted_tokens": 7540,
      "interactive_admitted": 4,
      "batch_admitted": 1,
      "rejected": [
        "b2",
        "b3"
      ],
      "tenant_tokens": {
  

## 9. Make the bounded decision

> Token-aware admission better represents inference cost than request counts, but production limits require native traffic calibration.

**Acceptance/rollback:** Adopt a policy only when premium SLOs, batch throughput, fairness, abuse resistance, and usage reconciliation meet written gates.

**Failure analysis:** Clients can understate output demand, tokenization varies by model, and retries can amplify load. A numerical queue omits cache and real scheduler interactions.


## 10. Extend the evidence

Place the gateway before a staging vLLM pool, replay signed multi-tenant traffic, cancel requests, exhaust quotas, and reconcile server usage with billing records.

The full boundary and references are in [`README.md`](README.md).
